In [1]:
import os

# Même correction que la Session 1 : VSCode connecté à un serveur Jupyter
# distant peut démarrer le kernel avec le mauvais dossier de travail.
# Cette cellule force le bon cwd avant tout, comme dans les Sessions 3 et 4.
if os.path.basename(os.getcwd()) != "notebooks":
    os.chdir("/home/jovyan/notebooks")

print("cwd:", os.getcwd())

cwd: /home/jovyan/notebooks


---
# PARTIE 2 -- L'API DataFrame

## 2.1 Du RDD au DataFrame

L'API RDD est puissante mais verbeuse et peu optimisee : Spark ne sait pas ce que
contiennent les elements (ils sont des `object` Python opaques). L'API **DataFrame**
introduit un schema explicite, ce qui permet a l'optimiseur Catalyst de generer
un plan d'execution bien plus efficace.

### Creation depuis un RDD


---
> **Transparence sur l'usage d'un assistant IA**
>
> Ce notebook a été relu et corrigé avec l'aide de l'assistant IA
> **Claude / Claude Code** (Anthropic) -- essentiellement des corrections
> ponctuelles (téléchargements rendus idempotents, variable manquante,
> problème d'environnement Docker), signalées individuellement par des
> commentaires à la première personne. Voir la section *« Transparence sur
> l'utilisation d'un assistant IA »* du `README.md` du projet pour le détail.
---


In [2]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
from pathlib import Path
import time

DATA_DIR      = Path("../data")
VELIB_RAW_DIR = DATA_DIR / "velib" / "raw"
VELIB_PARQ_DIR = DATA_DIR / "velib" / "parquet"   # j'ai ajouté cette ligne, absente ici mais utilisée en 2.2
METEO_DIR     = DATA_DIR / "meteo"
OUTPUT_DIR    = DATA_DIR / "output"

spark = (
    SparkSession.builder
    .appName("ClimaCity-Paris")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext
print("Spark prêt.")

Spark prêt.


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, BooleanType
)

schema_velib = StructType([
    StructField("horodatage",   StringType(),  nullable=True),
    StructField("capacite",     IntegerType(), nullable=True),
    StructField("velos_meca",   IntegerType(), nullable=True),
    StructField("velos_elec",   IntegerType(), nullable=True),
    StructField("nom_station",  StringType(),  nullable=True),
    StructField("coordonnees",  StringType(),  nullable=True),
    StructField("is_installed", BooleanType(), nullable=True),
])

df_velib = (
    spark.read
    .option("header", "false")
    .option("sep", ",")
    .schema(schema_velib)
    .csv(str(VELIB_RAW_DIR / "historique_stations.csv"))
)

df_velib.printSchema()
df_velib.show(5, truncate=False)
print(f"Lignes : {df_velib.count():,}  --  Partitions : {df_velib.rdd.getNumPartitions()}")

root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- is_installed: boolean (nullable = true)

+-----------------+--------+----------+----------+-----------------------------------+----------------+------------+
|horodatage       |capacite|velos_meca|velos_elec|nom_station                        |coordonnees     |is_installed|
+-----------------+--------+----------+----------+-----------------------------------+----------------+------------+
|2020-11-26T12:59Z|35      |4         |5         |Benjamin Godard - Victor Hugo      |48.86598,2.27572|true        |
|2020-11-26T12:59Z|55      |23        |4         |André Mazet - Saint-André des Arts |48.85376,2.33910|true        |
|2020-11-26T12:59Z|20      |0         |0         |Charonne - Robert et Sonia Delauney|48.85591,2.39257|true  

---
## 2.2 Lecture directe depuis les fichiers Parquet

En pratique, on ne passe presque jamais par les RDD pour creer un DataFrame.
Spark peut lire directement de nombreux formats (CSV, JSON, Parquet, Delta, ORC...).

Le format **Parquet** est le format de facto pour les donnees analytiques :
- Stockage colonnaire (lecture partielle possible)
- Compression integree (Snappy, Zstandard...)
- Schema embarque dans les fichiers
- Partitionnement natif (Hive-style)


In [4]:
# Lecture des fichiers Parquet pré-préparés (partitionnés par année et mois)
# Spark détecte automatiquement le schéma et les colonnes de partition
#
# J'ai dû adapter cette cellule : le jeu de données "pré-préparé" mentionné
# dans l'énoncé n'existe nulle part dans le projet (data/velib/parquet est vide,
# rien ne l'alimente ailleurs dans le notebook). Je le génère donc moi-même une
# fois, à partir du df_velib chargé en 2.1, partitionné par année/mois extraits
# de horodatage -- c'est exactement ce que cette section est censée illustrer
# (lecture Parquet partitionnée + predicate pushdown, cf. cellule suivante).
if not any(VELIB_PARQ_DIR.glob("annee=*")):
    print(f"{VELIB_PARQ_DIR} est vide -- génération du jeu de données pré-partitionné...")
    (
        df_velib
        .withColumn("annee", F.year(F.to_timestamp("horodatage", "yyyy-MM-dd'T'HH:mmX")))
        .withColumn("mois",  F.month(F.to_timestamp("horodatage", "yyyy-MM-dd'T'HH:mmX")))
        .write
        .mode("overwrite")
        .partitionBy("annee", "mois")
        .parquet(str(VELIB_PARQ_DIR))
    )
    print("Génération terminée.")

df_velib = spark.read.parquet(str(VELIB_PARQ_DIR))

df_velib.printSchema()
print(f"\nDimensions : {df_velib.count():,} lignes x {len(df_velib.columns)} colonnes")
print(f"Partitions : {df_velib.rdd.getNumPartitions()}")

root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- is_installed: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)


Dimensions : 10,986,730 lignes x 9 colonnes
Partitions : 5


In [5]:
# Lecture sélective d'une partition (predicate pushdown)
# Spark ne lira que les fichiers du dossier annee=2021/mois=2
#
# J'ai adapté l'exemple : nos données couvrent novembre 2020 à avril 2021,
# pas 2023 -- j'ai choisi une partition qui existe réellement (février 2021).
df_fevrier_2021 = spark.read.parquet(str(VELIB_PARQ_DIR)).filter("annee = 2021 AND mois = 2")
print(f"Février 2021 : {df_fevrier_2021.count():,} lignes")

# Comparer avec une lecture complète puis filtre -- le plan d'exécution est identique
# grâce au predicate pushdown, mais on le vérifie avec explain()
print("\n--- Plan d'exécution (filtre après lecture) ---")
df_velib.filter("annee = 2021 AND mois = 2").explain(mode="formatted")

Février 2021 : 2,230,346 lignes

--- Plan d'exécution (filtre après lecture) ---
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [9]: [horodatage#70, capacite#71, velos_meca#72, velos_elec#73, nom_station#74, coordonnees#75, is_installed#76, annee#77, mois#78]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/data/velib/parquet]
PartitionFilters: [isnotnull(annee#77), isnotnull(mois#78), (annee#77 = 2021), (mois#78 = 2)]
ReadSchema: struct<horodatage:string,capacite:int,velos_meca:int,velos_elec:int,nom_station:string,coordonnees:string,is_installed:boolean>

(2) ColumnarToRow [codegen id : 1]
Input [9]: [horodatage#70, capacite#71, velos_meca#72, velos_elec#73, nom_station#74, coordonnees#75, is_installed#76, annee#77, mois#78]




---
## 2.3 Exploration et diagnostic des donnees

Avant tout traitement, il faut comprendre la structure et la qualite des donnees.
Spark propose des fonctions d'agregation statistiques integrees.


In [6]:
from pyspark.sql import functions as F

# Vue d'ensemble statistique
print("=== Statistiques descriptives (colonnes numeriques) ===")
df_velib.describe("capacite", "velos_meca", "velos_elec").show()

=== Statistiques descriptives (colonnes numeriques) ===
+-------+------------------+-----------------+------------------+
|summary|          capacite|       velos_meca|        velos_elec|
+-------+------------------+-----------------+------------------+
|  count|          10986730|         10986730|          10986730|
|   mean|31.465765700986555|7.845410599878217|3.5020160684753336|
| stddev|12.018012456831205|9.007581146142359| 3.144554925684188|
|    min|                 0|                0|                 0|
|    max|                74|               73|                41|
+-------+------------------+-----------------+------------------+



In [7]:
# Comptage des valeurs nulles par colonne
print("=== Valeurs nulles par colonne ===")
df_velib.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_velib.columns
]).show()

=== Valeurs nulles par colonne ===
+----------+--------+----------+----------+-----------+-----------+------------+-----+----+
|horodatage|capacite|velos_meca|velos_elec|nom_station|coordonnees|is_installed|annee|mois|
+----------+--------+----------+----------+-----------+-----------+------------+-----+----+
|         0|       0|         0|         0|          0|          0|           0|    0|   0|
+----------+--------+----------+----------+-----------+-----------+------------+-----+----+



In [8]:
# Detection des anomalies evidentes
print("=== Anomalies detectees ===")

# Stations avec capacite <= 0
n_cap_zero = df_velib.filter(F.col("capacite") <= 0).count()
print(f"  Snapshots avec capacite <= 0          : {n_cap_zero:,}")

n_debord = df_velib.filter(
    (F.col("velos_meca") + F.col("velos_elec")) > (F.col("capacite") + 2)
).count()
print(f"  Snapshots avec total > capacite + 2   : {n_debord:,}")

for col_name in ["velos_meca", "velos_elec"]:
    n_neg = df_velib.filter(F.col(col_name) < 0).count()
    print(f"  {col_name:<30} < 0 : {n_neg:,}")


=== Anomalies detectees ===
  Snapshots avec capacite <= 0          : 64,762
  Snapshots avec total > capacite + 2   : 42
  velos_meca                     < 0 : 0
  velos_elec                     < 0 : 0


In [9]:
# Distribution temporelle : combien de snapshots par mois ?
print("=== Couverture temporelle ===")
(
df_velib
.withColumn("mois", F.substring(F.col("horodatage"), 1, 7))
.groupBy("mois")
.count()
.orderBy("mois")
.show(50)
)


=== Couverture temporelle ===
+-------+-------+
|   mois|  count|
+-------+-------+
|2020-11| 549021|
|2020-12|2809367|
|2021-01|1515745|
|2021-02|2230346|
|2021-03|2861805|
|2021-04|1020446|
+-------+-------+



---
## 2.4 Nettoyage et construction des features

Nous allons maintenant construire la table `disponibilite` propre, telle que
definie dans le schema cible du projet.


In [10]:
from pyspark.sql.functions import (
    to_timestamp, col, when, round as spark_round,
    year, month, dayofweek, hour, expr
)

df_clean = (
    df_velib
    # 1. Parsing du timestamp
    .withColumn("horodatage", to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mmX"))
    # 2. Supprimer les timestamps invalides
    .filter(col("horodatage").isNotNull())
    # 3. Supprimer les capacités invalides
    .filter(col("capacite") > 0)
    # 4. Corriger les valeurs négatives
    .withColumn("velos_meca", when(col("velos_meca") < 0, 0).otherwise(col("velos_meca")))
    .withColumn("velos_elec", when(col("velos_elec") < 0, 0).otherwise(col("velos_elec")))
    # 5. Calcul du taux d'occupation
    .withColumn("velos_total", col("velos_meca") + col("velos_elec"))
    .withColumn("taux_occupation", spark_round(col("velos_total") / col("capacite"), 4))
    # 6. Bornage entre 0 et 1
    .withColumn("taux_occupation", expr("greatest(0.0, least(1.0, taux_occupation))"))
    # 7. Features temporelles
    .withColumn("annee",      year(col("horodatage")))
    .withColumn("mois",       month(col("horodatage")))
    .withColumn("jour_sem",   dayofweek(col("horodatage")))
    .withColumn("heure",      hour(col("horodatage")))
    .withColumn("is_weekend", col("jour_sem").isin(1, 7))
)

print(f"Lignes après nettoyage : {df_clean.count():,}")
df_clean.printSchema()
df_clean.show(3, truncate=True)

Lignes après nettoyage : 10,921,968
root
 |-- horodatage: timestamp (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- is_installed: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- velos_total: integer (nullable = true)
 |-- taux_occupation: double (nullable = false)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)

+-------------------+--------+----------+----------+--------------------+----------------+------------+-----+----+-----------+---------------+--------+-----+----------+
|         horodatage|capacite|velos_meca|velos_elec|         nom_station|     coordonnees|is_installed|annee|mois|velos_total|taux_occupation|jour_sem|heure|is_weekend|
+-------------------+-------

In [11]:
# [EXERCICE]
# Ajoutez une colonne "statut" de type StringType avec les valeurs :
#   "vide"   si taux_occupation < 0.1
#   "plein"  si taux_occupation > 0.9
#   "normal" sinon
#
# Indice : utilisez F.when(...).when(...).otherwise(...)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
df_clean = df_clean.withColumn(
    "statut",
    # ...
F.when(F.col("taux_occupation") < 0.1, "vide")
 .when(F.col("taux_occupation") > 0.9, "plein")
 .otherwise("normal")
)

df_clean.groupBy("statut").count().orderBy("statut").show()


+------+-------+
|statut|  count|
+------+-------+
|normal|8618099|
| plein| 428471|
|  vide|1875398|
+------+-------+



---
## 2.5 Chargement des donnees meteorologiques

La source meteorologique est le fichier SYNOP de Meteo-France pour la station
Paris-Montsouris, telechargeable librement sur data.gouv.fr.
Le format SYNOP est semi-structure et necessite quelques transformations.

Voici le format des colonnes pertinentes :

| Colonne SYNOP | Signification | Unite |
|---------------|---------------|-------|
| `AAAAMMJJHH`  | Horodatage    | UTC   |
| `T`           | Temperature   | K (Kelvin) |
| `U`           | Humidite relative | % |
| `FF`          | Vitesse du vent | m/s |
| `RR1`         | Precipitation | mm/h |
| `N`           | Nebulosity (couverture nuageuse) | octals (0-8) |


In [12]:
import urllib.request
import json

METEO_CSV = METEO_DIR / "paris_montsouris_horaire.csv"
METEO_DIR.mkdir(parents=True, exist_ok=True)

# J'ai ajouté cette vérification, comme pour les téléchargements de la Session 1 :
# pas besoin de rappeler l'API Open-Meteo si le fichier est déjà là.
if METEO_CSV.exists():
    print(f"[OK] {METEO_CSV} existe déjà -- téléchargement ignoré.")
else:
    # API Open-Meteo : données horaires Paris 2020-2021
    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        "latitude=48.8236&longitude=2.3378"
        "&start_date=2020-11-01&end_date=2021-04-30"
        "&hourly=temperature_2m,relativehumidity_2m,windspeed_10m,precipitation"
        "&timezone=Europe%2FParis"
    )

    print("Téléchargement des données météo...")
    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read())

    hourly = data["hourly"]
    import csv
    with open(METEO_CSV, "w", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(["NUM_POSTE","NOM_USUEL","LAT","LON","ALTI","AAAAMMJJHH","T","U","FF","RR1","N"])
        for i, ts in enumerate(hourly["time"]):
            dt = ts.replace("-","").replace("T","").replace(":","")[:10]
            writer.writerow([
                "75114001", "PARIS-MONTSOURIS", 48.8236, 2.3378, 75,
                dt,
                hourly["temperature_2m"][i],
                hourly["relativehumidity_2m"][i],
                hourly["windspeed_10m"][i],
                hourly["precipitation"][i],
                None
            ])

    print(f"Météo sauvegardée : {METEO_CSV}")

[OK] ../data/meteo/paris_montsouris_horaire.csv existe déjà -- téléchargement ignoré.


In [13]:
# Lecture du CSV avec separateur point-virgule
# Le fichier contient des en-tetes sur plusieurs lignes -- on saute les deux premieres
schema_meteo = StructType([
    StructField("NUM_POSTE",   StringType(),  True),
    StructField("NOM_USUEL",   StringType(),  True),
    StructField("LAT",         DoubleType(),  True),
    StructField("LON",         DoubleType(),  True),
    StructField("ALTI",        DoubleType(),  True),
    StructField("AAAAMMJJHH",  StringType(),  True),
    StructField("T",           DoubleType(),  True),
    StructField("U",           DoubleType(),  True),
    StructField("FF",          DoubleType(),  True),
    StructField("RR1",         DoubleType(),  True),
    StructField("N",           DoubleType(),  True),
])

df_meteo_raw = (
    spark.read
    .option("sep", ";")
    .option("header", True)
    .option("nullValue", "mq")    # "mq" = manquant dans SYNOP
    .schema(schema_meteo)
    .csv(str(METEO_CSV))
)

print(f"Lignes meteo brutes : {df_meteo_raw.count():,}")
df_meteo_raw.show(5)

Lignes meteo brutes : 4,344
+---------+----------------+-------+------+----+----------+----+----+----+---+----+
|NUM_POSTE|       NOM_USUEL|    LAT|   LON|ALTI|AAAAMMJJHH|   T|   U|  FF|RR1|   N|
+---------+----------------+-------+------+----+----------+----+----+----+---+----+
| 75114001|PARIS-MONTSOURIS|48.8236|2.3378|75.0|2020110100|14.5|87.0|15.2|0.0|NULL|
| 75114001|PARIS-MONTSOURIS|48.8236|2.3378|75.0|2020110101|14.1|84.0|14.7|0.0|NULL|
| 75114001|PARIS-MONTSOURIS|48.8236|2.3378|75.0|2020110102|13.3|87.0|14.2|0.0|NULL|
| 75114001|PARIS-MONTSOURIS|48.8236|2.3378|75.0|2020110103|13.3|87.0|12.3|0.0|NULL|
| 75114001|PARIS-MONTSOURIS|48.8236|2.3378|75.0|2020110104|13.1|90.0|10.2|0.1|NULL|
+---------+----------------+-------+------+----+----------+----+----+----+---+----+
only showing top 5 rows



In [14]:
from pyspark.sql.functions import (
    to_timestamp, substring, concat_ws,
    lit, lpad
)

df_meteo = (
    df_meteo_raw
    .withColumn("horodatage_meteo", to_timestamp(col("AAAAMMJJHH"), "yyyyMMddHH"))
    .select(
        col("horodatage_meteo"),
        col("T").alias("temperature_c"),
        col("U").alias("humidite_pct"),
        col("FF").alias("vent_kmh"),
        col("RR1").alias("precipitation_mm"),
    )
    .filter(col("horodatage_meteo").isNotNull())
)

df_meteo = df_meteo.withColumn(
    "est_pluie", col("precipitation_mm") > 0.5
)

print(f"Lignes météo nettoyées : {df_meteo.count():,}")
df_meteo.show(8)
df_meteo.describe(["temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"]).show()

Lignes météo nettoyées : 4,344
+-------------------+-------------+------------+--------+----------------+---------+
|   horodatage_meteo|temperature_c|humidite_pct|vent_kmh|precipitation_mm|est_pluie|
+-------------------+-------------+------------+--------+----------------+---------+
|2020-11-01 00:00:00|         14.5|        87.0|    15.2|             0.0|    false|
|2020-11-01 01:00:00|         14.1|        84.0|    14.7|             0.0|    false|
|2020-11-01 02:00:00|         13.3|        87.0|    14.2|             0.0|    false|
|2020-11-01 03:00:00|         13.3|        87.0|    12.3|             0.0|    false|
|2020-11-01 04:00:00|         13.1|        90.0|    10.2|             0.1|    false|
|2020-11-01 05:00:00|         13.1|        94.0|    10.3|             0.6|     true|
|2020-11-01 06:00:00|         14.0|        92.0|    16.1|             0.5|    false|
|2020-11-01 07:00:00|         14.5|        94.0|    17.9|             0.0|    false|
+-------------------+-------------

---
## 2.6 Jointure temporelle : Velib' x Meteo

Les observations Velib' sont a la minute, les donnees SYNOP a l'heure.
Pour joindre les deux tables, on **tronque** les horodatages Velib' a l'heure la plus
proche, puis on effectue une jointure sur cette cle commune.


In [15]:
from pyspark.sql.functions import date_trunc, broadcast

df_velib_join = df_clean.withColumn(
    "heure_tronquee",
    date_trunc("hour", col("horodatage"))
)

df_meteo_join = df_meteo.withColumnRenamed(
    "horodatage_meteo", "heure_tronquee"
)

df_joint = (
    df_velib_join
    .join(broadcast(df_meteo_join), on="heure_tronquee", how="left")
)

n_avant      = df_clean.count()
n_apres      = df_joint.count()
n_meteo_null = df_joint.filter(col("temperature_c").isNull()).count()

print(f"Lignes avant jointure  : {n_avant:,}")
print(f"Lignes après jointure  : {n_apres:,}")
print(f"Snapshots sans météo   : {n_meteo_null:,} ({100*n_meteo_null/n_apres:.1f}%)")

Lignes avant jointure  : 10,921,968
Lignes après jointure  : 10,921,968
Snapshots sans météo   : 0 (0.0%)


In [16]:
# Examinons le plan d'execution pour verifier que le broadcast est bien utilise
print("=== Plan d'execution de la jointure ===")
df_joint.explain(mode="formatted")
# Cherchez "BroadcastHashJoin" dans le plan -- pas de "SortMergeJoin" (qui implique un shuffle)


=== Plan d'execution de la jointure ===
== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- BroadcastHashJoin LeftOuter BuildRight (11)
      :- Project (6)
      :  +- Project (5)
      :     +- Project (4)
      :        +- Project (3)
      :           +- Filter (2)
      :              +- Scan parquet  (1)
      +- BroadcastExchange (10)
         +- Project (9)
            +- Filter (8)
               +- Scan csv  (7)


(1) Scan parquet 
Output [9]: [horodatage#70, capacite#71, velos_meca#72, velos_elec#73, nom_station#74, coordonnees#75, is_installed#76, annee#77, mois#78]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/data/velib/parquet]
PushedFilters: [IsNotNull(capacite), GreaterThan(capacite,0)]
ReadSchema: struct<horodatage:string,capacite:int,velos_meca:int,velos_elec:int,nom_station:string,coordonnees:string,is_installed:boolean>

(2) Filter
Input [9]: [horodatage#70, capacite#71, velos_meca#72, velos_elec#73, nom_station#74, coordonnees#75, is_in

In [17]:
# [EXERCICE]
# Calculez, pour chaque combinaison (arrondissement, est_pluie),
# le taux_occupation moyen et le nombre de snapshots.
# Triez par arrondissement croissant, puis par est_pluie.
#
# Indice : utilisez groupBy(...).agg(F.mean(...).alias(...), F.count("*")...)
# ──────────────────────────────────────────────────────────────────────────

(
    df_joint
    .groupBy("nom_station", "est_pluie")
    .agg(
        F.round(F.mean("taux_occupation"), 4).alias("taux_moyen"),
        F.count("*").alias("nb_snapshots")
    )
    .orderBy("nom_station", "est_pluie")
    .show(20)
)


+--------------------+---------+----------+------------+
|         nom_station|est_pluie|taux_moyen|nb_snapshots|
+--------------------+---------+----------+------------+
| Jean Bleuzen - S...|    false|    0.1013|        7507|
| Jean Bleuzen - S...|     true|    0.1109|         359|
| Place Léon Gambetta|    false|    0.5852|        7507|
| Place Léon Gambetta|     true|    0.5658|         359|
|11 Novembre 1918 ...|    false|    0.0658|        7507|
|11 Novembre 1918 ...|     true|    0.0862|         359|
|18 juin 1940 - Bu...|    false|    0.1977|        7507|
|18 juin 1940 - Bu...|     true|    0.1769|         359|
|8 Mai 1945 - 10 J...|    false|       0.0|        7507|
|8 Mai 1945 - 10 J...|     true|       0.0|         359|
|Abbeville - Faubo...|    false|    0.3824|        7507|
|Abbeville - Faubo...|     true|    0.4133|         359|
|Abbé Carton - Pla...|    false|    0.2009|        7507|
|Abbé Carton - Pla...|     true|    0.2631|         359|
|Abbé Groult - Con...|    false

---
## 2.7 Persistance en memoire : `.cache()` et `.persist()`

Nous allons utiliser `df_joint` intensivement pendant le reste du cours.
Mettons-le en cache pour eviter de recalculer la jointure a chaque action.


In [18]:
# .cache() == .persist(StorageLevel.MEMORY_AND_DISK)
# Si les donnees ne tiennent pas en RAM, Spark deborde sur disque.

t0 = time.perf_counter()
df_joint.cache()
df_joint.count()   # force la materialisation du cache
t_cache = time.perf_counter() - t0
print(f"Mise en cache (premiere passe) : {t_cache:.2f} s")

# Deuxieme passe -- depuis le cache
t0 = time.perf_counter()
df_joint.count()
t_hit = time.perf_counter() - t0
print(f"Lecture depuis le cache        : {t_hit:.2f} s")
print(f"Gain                           : x{t_cache/t_hit:.1f}")
print()
print("Allez dans Spark UI -> Storage pour voir la taille du cache.")


Mise en cache (premiere passe) : 12.76 s
Lecture depuis le cache        : 0.07 s
Gain                           : x186.9

Allez dans Spark UI -> Storage pour voir la taille du cache.


In [19]:
# StorageLevel disponibles (par ordre de rapidite / consommation memoire)
from pyspark import StorageLevel

print("StorageLevels disponibles :")
print("  MEMORY_ONLY          : RAM uniquement (eviction si plein)")
print("  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]")
print("  DISK_ONLY            : Disque uniquement (lent mais stable)")
print("  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)")
print("  OFF_HEAP             : Memoire hors JVM (necessite configuration)")
print()
print("Regle pratique :")
print("  - Petits DataFrames reutilises souvent -> MEMORY_ONLY")
print("  - Gros DataFrames reutilises -> MEMORY_AND_DISK")
print("  - Ne pas cacher si utilise une seule fois -> overhead inutile")


StorageLevels disponibles :
  MEMORY_ONLY          : RAM uniquement (eviction si plein)
  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]
  DISK_ONLY            : Disque uniquement (lent mais stable)
  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)
  OFF_HEAP             : Memoire hors JVM (necessite configuration)

Regle pratique :
  - Petits DataFrames reutilises souvent -> MEMORY_ONLY
  - Gros DataFrames reutilises -> MEMORY_AND_DISK
  - Ne pas cacher si utilise une seule fois -> overhead inutile


---
## 2.8 Ecriture en Parquet partitionne

La table `df_joint` est la table centrale du projet. Nous allons la persister
sur disque en Parquet, partitionne par annee et par mois, pour que les
traitements des jours suivants n'aient pas a la reconstruire.


In [20]:
OUTPUT_VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"

colonnes_finales = [
    "nom_station", "capacite",
    "horodatage", "velos_meca", "velos_elec", "velos_total",
    "taux_occupation", "statut",
    "annee", "mois", "jour_sem", "heure", "is_weekend",
    "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm", "est_pluie"
]

df_sortie = df_joint.select(*colonnes_finales)

t0 = time.perf_counter()
(
    df_sortie
    .write
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .parquet(str(OUTPUT_VELIB_CONSOLIDE))
)
t_write = time.perf_counter() - t0
print(f"Écriture en {t_write:.1f} s")

# Vérification
df_verif = spark.read.parquet(str(OUTPUT_VELIB_CONSOLIDE))
print(f"Total relu : {df_verif.count():,} lignes")
print(f"Colonnes   : {df_verif.columns}")
df_verif.show(3)

Écriture en 8.5 s
Total relu : 10,921,968 lignes
Colonnes   : ['nom_station', 'capacite', 'horodatage', 'velos_meca', 'velos_elec', 'velos_total', 'taux_occupation', 'statut', 'jour_sem', 'heure', 'is_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'annee', 'mois']
+--------------------+--------+-------------------+----------+----------+-----------+---------------+------+--------+-----+----------+-------------+------------+--------+----------------+---------+-----+----+
|         nom_station|capacite|         horodatage|velos_meca|velos_elec|velos_total|taux_occupation|statut|jour_sem|heure|is_weekend|temperature_c|humidite_pct|vent_kmh|precipitation_mm|est_pluie|annee|mois|
+--------------------+--------+-------------------+----------+----------+-----------+---------------+------+--------+-----+----------+-------------+------------+--------+----------------+---------+-----+----+
|Maxime Gorki - Lo...|      20|2021-02-16 05:48:00|         0|     

In [21]:
# Comparaison de la taille CSV vs Parquet
import os

def taille_dossier_mb(path: Path) -> float:
    """
    Calcule la taille d'un dossier en octets.
    """
def taille_dossier_mb(path: Path) -> float:
    """Calcule la taille totale d'un dossier en MB."""
    total = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fichiers in os.walk(path)
        for f in fichiers
    )
    return total / 1_048_576
t_csv   = taille_dossier_mb(VELIB_RAW_DIR)
t_parq  = taille_dossier_mb(OUTPUT_VELIB_CONSOLIDE)
ratio   = t_csv / t_parq if t_parq > 0 else 0

print(f"Taille CSV bruts (compresses .gz) : {t_csv:.1f} MB")
print(f"Taille Parquet consolide          : {t_parq:.1f} MB")
print(f"Rapport CSV/Parquet               : x{ratio:.1f}")
print()
print("Le Parquet est plus petit car :")
print("  1. Stockage colonnaire : on compresse des valeurs homogenes")
print("  2. Encodage par dictionnaire pour les strings repetitives (noms de stations)")
print("  3. Run-Length Encoding sur les colonnes de partition (annee, mois)")


Taille CSV bruts (compresses .gz) : 780.0 MB
Taille Parquet consolide          : 56.0 MB
Rapport CSV/Parquet               : x13.9

Le Parquet est plus petit car :
  1. Stockage colonnaire : on compresse des valeurs homogenes
  2. Encodage par dictionnaire pour les strings repetitives (noms de stations)
  3. Run-Length Encoding sur les colonnes de partition (annee, mois)


---
## Bilan du Jour 1

### Ce que nous avons fait

| Etape | API | Concept cle |
|-------|-----|-------------|
| Chargement CSV brut | RDD | `sc.textFile()`, parsing manuel |
| Filtrage et transformation | RDD | `map()`, `filter()`, evaluation paresseuse |
| Agregation par cle | RDD | `reduceByKey()` vs `groupByKey()` |
| Profil horaire | RDD | `flatMap()`, calcul de moyenne distribuee |
| Jointure RDD | RDD | `join()` sur paires (cle, valeur) |
| Lecture Parquet | DataFrame | `spark.read.parquet()`, predicate pushdown |
| Exploration statistique | DataFrame | `describe()`, comptage de nulls |
| Nettoyage | DataFrame | `dropna()`, `when()`, `greatest()` |
| Features temporelles | DataFrame | `year()`, `hour()`, `dayofweek()` |
| Jointure broadcast | DataFrame | `broadcast()`, eviter le shuffle |
| Persistance | DataFrame/RDD | `.cache()`, `.persist()`, `StorageLevel` |
| Ecriture partitionnee | DataFrame | `write.partitionBy().parquet()` |

### Concepts fondamentaux a retenir

1. **Evaluation paresseuse** : les transformations construisent un plan, les actions l'executent.
2. **Le DAG** : lire le Spark UI est une competence essentielle, pas optionnelle.
3. **reduceByKey > groupByKey** : toujours combiner localement avant de shuffler.
4. **DataFrame > RDD** : l'optimiseur Catalyst rend les DataFrames significativement plus
   rapides que les RDD pour les memes operations.
5. **Cache strategique** : ne mettre en cache que ce qui est reutilise plusieurs fois.
6. **Broadcast join** : quand une table est petite, eviter le shuffle de la grosse table.

### Pour demain (Jour 2)

La table `disponibilite_consolidee.parquet` sera le point de depart du Jour 2.
Nous allons l'interroger avec Spark SQL (fenetrage temporel, requetes analytiques),
puis la connecter a un flux simule de mises a jour en temps reel avec Structured Streaming.


In [22]:
# Nettoyage de fin de session
# Toujours liberer la SparkSession proprement pour liberer les ressources
print("Arret de la SparkSession...")
spark.stop()
print("SparkSession arretee. Bonne nuit !")


Arret de la SparkSession...
SparkSession arretee. Bonne nuit !
